In [1]:
import numpy as np
from scipy.optimize import minimize
from scipy.stats import norm

from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import (
    ConstantKernel,
    Matern,
    WhiteKernel,
)

In [2]:
# ==========================================
# Your existing data
# X 4D (30,4)
# y 1D (30,)
# ==========================================

# Example
X = np.load('function_4/initial_inputs.npy')
y = np.load('function_4/initial_outputs.npy')

X = np.vstack([X, [0.440401, 0.425456, 0.378309, 0.397056],
                  [0.409358, 0.417289, 0.352664, 0.435519],
                  [0.428108, 0.421562, 0.301114, 0.432399],
                  [0.380951, 0.416208, 0.421633, 0.438489],
                  [0.393132, 0.414796, 0.408439, 0.433212],
                  [0.395565, 0.414772, 0.406086, 0.432731],
                  [0.397142, 0.415299, 0.404123, 0.432599],
                  [0.397889, 0.416821, 0.402223, 0.432633],
                  [0.398342, 0.417447, 0.400213, 0.432772],
                  [0.406749, 0.398113, 0.399310, 0.436798],
                  [0.381215, 0.450967, 0.398693, 0.425951]])
                  
y = np.append(y, [0.26046255285558617, 
                  0.5378997502104821,
                  -0.5542192005550493,
                  0.43093188480821,
                  0.4752383673405878,
                  0.4742379750281853,
                  0.4711373374072889,
                  0.46598200600632866,
                  0.452701373792483,
                  0.26299494478158225,
                  -0.09525663073301471])

In [3]:
# ==========================================
# Fit Gaussian Process
# ==========================================

kernel = (
    ConstantKernel(1.0)
    * Matern(length_scale=np.ones(X.shape[1]), nu=2.5)
    + WhiteKernel(noise_level=1e-6)
)

gp = GaussianProcessRegressor(
    kernel=kernel,
    normalize_y=True,
    n_restarts_optimizer=10,
    random_state=0
)

gp.fit(X, y)

# ==========================================
# Expected Improvement
# ==========================================

def expected_improvement(x, gp, y_best, xi=0.001):
    x = np.array(x).reshape(1, -1)

    mu, sigma = gp.predict(x, return_std=True)

    sigma = sigma[0]
    mu = mu[0]

    if sigma < 1e-12:
        return 0.0

    improvement = mu - y_best - xi
    Z = improvement / sigma

    ei = improvement * norm.cdf(Z) + sigma * norm.pdf(Z)

    return ei



# ==========================================
# Optimise acquisition function
# ==========================================

bounds = list(zip(X.min(axis=0), X.max(axis=0))) # replace with your limits

y_best = np.max(y)


def objective(x):
    return -expected_improvement(x, gp, y_best)


best_x = None
best_ei = -np.inf

for _ in range(10000):

    x0 = np.random.uniform(
        [b[0] for b in bounds],
        [b[1] for b in bounds]
    )

    res = minimize(
        objective,
        x0=x0,
        bounds=bounds,
        method="L-BFGS-B"
    )

    ei = -res.fun

    if ei > best_ei:
        best_ei = ei
        best_x = res.x

print("Next point to evaluate:")
next_point = np.round(best_x, decimals=6)
print(next_point)

print("Expected Improvement:")
print(best_ei)

Next point to evaluate:
[0.408596 0.382761 0.403825 0.434602]
Expected Improvement:
0.05202662505537756
